# Assignment 2 — Deep Learning (NNDL)

> Spec PDF: `/Users/tahamajs/Documents/uni/LLM/Deep_UT/This_year/CA2/description/NNDL_Assignment2.pdf`

This notebook contains complete, runnable code and organized report sections for both questions. Fill in the textual answers under each subsection and run the code cells to produce your results.

In [ ]:
# Environment and Imports
import warnings
warnings.filterwarnings('ignore')

import os
import math
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms, datasets, models

from sklearn.metrics import confusion_matrix, classification_report

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)

device = torch.device('cpu')
if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')

print(f"Using device: {device}")

# Table of Contents
#
- [Question 1 — Rice Leaf Disease Detection with CNN](#question-1--rice-leaf-disease-detection-with-cnn)
  - [1-1. Dataset Preparation (5 pts)](#1-1-dataset-preparation-5-pts)
  - [1-2. Data Preprocessing (20 pts)](#1-2-data-preprocessing-20-pts)
  - [1-3. AlexNet Model](#1-3-alexnet-model)
    - [1-3-1. Implementation (20 pts)](#1-3-1-implementation-20-pts)
    - [1-3-2. Results and Evaluation (15 pts)](#1-3-2-results-and-evaluation-15-pts)
  - [1-4. Paper’s Proposed Model (Transfer Learning)](#1-4-papers-proposed-model-transfer-learning)
    - [1-4-1. Implementation (20 pts)](#1-4-1-implementation-20-pts)
    - [1-4-2. Results and Evaluation (20 pts)](#1-4-2-results-and-evaluation-20-pts)
- [Question 2 — Vehicle Classification in Road Scenes](#question-2--vehicle-classification-in-road-scenes)
  - [2-1. Dataset Preparation (10 pts)](#2-1-dataset-preparation-10-pts)
  - [2-2. Data Preprocessing (15 pts)](#2-2-data-preprocessing-15-pts)
  - [2-3. Implementation (20 pts)](#2-3-implementation-20-pts)
  - [2-4. Evaluation (30 pts)](#2-4-evaluation-30-pts)
  - [2-5. Analysis and Optimization](#2-5-analysis-and-optimization)
    - [2-5-1. Effect of Data Augmentation (10)](#2-5-1-effect-of-data-augmentation-10)
    - [2-5-2. Role of Optimizer/Regularization (10)](#2-5-2-role-of-optimizerregularization-10)
    - [2-5-3. Fine-tuning (10)](#2-5-3-fine-tuning-10)
- [Submission Checklist](#submission-checklist)

# Notebook Overview
- Organized into clear sections per question with matching subsections to the spec (Dataset, Preprocessing, Implementation, Results, Analysis).
- Helpers (training/evaluation utilities) are centralized and referenced by both Q1 and Q2, similar to CA1’s structure.
- Use the checklists and templates to keep reporting concise and complete.

# Question 1 — Rice Leaf Disease Detection with CNN

> Paste the exact question prompt here if needed. This section follows the grading rubric outlined in the PDF.

## 1-1. Dataset Preparation (5 pts)
- Download dataset from Kaggle (Rice Disease Dataset).
- Split into train/val/test with 70/15/15 for 3 classes (Healthy + 3 disease classes).
- Plot class distributions per split (histogram/bar).

## 1-2. Data Preprocessing (20 pts)
- Resize all images to 224×224.
- Normalize pixel values to [0, 1] and apply ImageNet normalization for transfer learning.
- Apply at least 3 suitable augmentations (e.g., RandomResizedCrop, HorizontalFlip, ColorJitter). Justify choices.
- Show a few augmented samples for sanity check.

## 1-3. AlexNet Model

### 1-3-1. Implementation (20 pts)
- Implement AlexNet from scratch (PyTorch).
- Provide a model summary.
- Train for ~30 epochs; choose a reasonable optimizer, LR, weight decay; justify choices.

### 1-3-2. Results and Evaluation (15 pts)
- Plot training/validation loss and accuracy across epochs.
- Compute metrics on the test set: Accuracy, Sensitivity (Recall/TPR), Specificity (TNR).
- Report a confusion matrix and discuss which classes are most/least correctly classified and the common confusions.

## 1-4. Paper’s Proposed Model (Transfer Learning)

### 1-4-1. Implementation (20 pts)
- Implement the proposed architecture using VGG19 pretrained on ImageNet as base (freeze early layers).
- Re-do preprocessing to include ImageNet mean/std normalization for all eval sets.
- Keep number of epochs comparable to AlexNet for fair comparison.

### 1-4-2. Results and Evaluation (20 pts)
- Report training curves and test metrics as above.
- Compare against AlexNet:
  - Did the proposed architecture improve data efficiency (higher accuracy with fewer epochs)?
  - Explain how pretrained weights improved performance.
  - Design choices: why combining VGGNet end layers with Inception-like blocks?
  - Why use Global Pooling instead of fully connected layers at the end?

## Q1 — Overview
- Goal: classify rice leaf images into Healthy + 3 diseases.
- Models: AlexNet (from scratch) and a transfer learning model (VGG19).
- Report: accuracy, per-class sensitivity/specificity, confusion matrix; compare AlexNet vs TL.
- Guidance mirrors CA1 style: data prep → preprocessing → implementation → evaluation → analysis.

### 1-1. Dataset Preparation (5 pts)
- Download Rice Disease Dataset from Kaggle (Healthy + 3 diseases).
- Split: 70% train, 15% val, 15% test (stratified).
- Report class distributions and plot histogram per split.

In [ ]:
# Q1: Dataset Download (Rice Disease)
# Uncomment to download:
# import kagglehub
# path = kagglehub.dataset_download("anshulm257/rice-disease-dataset")
# print('Downloaded to:', path)

# Manual setup: Update paths below to your local dataset
RICE_DATA_ROOT = Path('/path/to/rice-disease-dataset')  # UPDATE THIS
RICE_CLASSES = ['Healthy', 'BacterialLeafBlight', 'BrownSpot', 'LeafSmut']  # Example; adjust per your data

# Check if dataset exists
if not RICE_DATA_ROOT.exists():
    print(f"Dataset not found at {RICE_DATA_ROOT}. Please download and update RICE_DATA_ROOT.")
else:
    print(f"Dataset root: {RICE_DATA_ROOT}")

In [ ]:
# Q1: Split dataset 70/15/15 and plot distributions
from sklearn.model_selection import train_test_split
import shutil

def split_rice_dataset(src_root, dest_root, classes, train_r=0.7, val_r=0.15, test_r=0.15, seed=42):
    """Split ImageFolder dataset into train/val/test."""
    dest_root = Path(dest_root)
    for split in ['train', 'val', 'test']:
        for cls in classes:
            (dest_root / split / cls).mkdir(parents=True, exist_ok=True)
    
    class_counts = {'train': {}, 'val': {}, 'test': {}}
    for cls in classes:
        cls_path = Path(src_root) / cls
        if not cls_path.exists():
            print(f"Class {cls} not found at {cls_path}")
            continue
        images = sorted(list(cls_path.glob('*.*')))
        train, temp = train_test_split(images, train_size=train_r, random_state=seed)
        val, test = train_test_split(temp, train_size=val_r/(val_r+test_r), random_state=seed)
        
        for img in train:
            shutil.copy(img, dest_root / 'train' / cls / img.name)
        for img in val:
            shutil.copy(img, dest_root / 'val' / cls / img.name)
        for img in test:
            shutil.copy(img, dest_root / 'test' / cls / img.name)
        
        class_counts['train'][cls] = len(train)
        class_counts['val'][cls] = len(val)
        class_counts['test'][cls] = len(test)
    return class_counts

# Example usage (uncomment when dataset is ready):
# RICE_SPLIT_ROOT = Path('rice_split')
# counts = split_rice_dataset(RICE_DATA_ROOT, RICE_SPLIT_ROOT, RICE_CLASSES)
# print('Split counts:', counts)

# Plot class distribution
def plot_split_distribution(counts, classes):
    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    for i, split in enumerate(['train', 'val', 'test']):
        vals = [counts[split].get(c, 0) for c in classes]
        axs[i].bar(classes, vals, color='steelblue')
        axs[i].set_title(f'{split.capitalize()} Distribution')
        axs[i].set_ylabel('Count')
        axs[i].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

# plot_split_distribution(counts, RICE_CLASSES)

### 1-2. Data Preprocessing (20 pts)
- Resize to 224×224.
- Normalize pixels [0,1]; for transfer learning, apply ImageNet mean/std.
- Implement ≥3 augmentations (RandomResizedCrop, HorizontalFlip, ColorJitter) and justify choices.
- Show augmented examples.

In [ ]:
# Q1: Show augmented samples
def show_augmented_samples(dataset_path, class_name, transform, n=8):
    """Display original + augmented versions of one image."""
    cls_path = Path(dataset_path) / class_name
    if not cls_path.exists():
        print(f"Class path {cls_path} not found.")
        return
    imgs = list(cls_path.glob('*.*'))
    if not imgs:
        print(f"No images in {cls_path}")
        return
    from PIL import Image
    img_path = imgs[0]
    img = Image.open(img_path).convert('RGB')
    
    fig, axs = plt.subplots(2, n//2, figsize=(12, 6))
    axs = axs.ravel()
    for i in range(n):
        aug_img = transform(img)
        if isinstance(aug_img, torch.Tensor):
            aug_img = aug_img.permute(1, 2, 0).numpy()
            aug_img = np.clip(aug_img, 0, 1)
        axs[i].imshow(aug_img)
        axs[i].axis('off')
        axs[i].set_title(f'Aug {i+1}')
    plt.suptitle(f'Augmented samples from class: {class_name}')
    plt.tight_layout()
    plt.show()

# Example (uncomment when dataset ready):
# train_tfms_q1, _ = get_transforms(224, for_transfer=False)
# show_augmented_samples(RICE_SPLIT_ROOT / 'train', RICE_CLASSES[0], train_tfms_q1)

### 1-3-1. AlexNet Implementation (20 pts)
- Implement AlexNet from scratch (see cell above for architecture).
- Train for 30 epochs; choose optimizer, LR, weight decay; justify.
- Provide model summary.

In [ ]:
# Q1: Train AlexNet (uncomment when dataset ready)
# CONFIG_Q1 = CONFIG.copy()
# CONFIG_Q1['task'] = 'rice'
# CONFIG_Q1['model'] = 'alexnet'
# CONFIG_Q1['epochs'] = 30
# CONFIG_Q1['data']['rice']['train_dir'] = str(RICE_SPLIT_ROOT / 'train')
# CONFIG_Q1['data']['rice']['val_dir'] = str(RICE_SPLIT_ROOT / 'val')
# CONFIG_Q1['data']['rice']['test_dir'] = str(RICE_SPLIT_ROOT / 'test')
# CONFIG_Q1['data']['rice']['num_classes'] = len(RICE_CLASSES)

# loaders_q1 = build_dataloaders(CONFIG_Q1)
# if loaders_q1:
#     train_loader_q1, val_loader_q1, test_loader_q1, class_names_q1 = loaders_q1
#     model_alex = AlexNet(num_classes=len(class_names_q1)).to(device)
#     print(model_alex)
#     history_alex, best_path_alex = fit(
#         model_alex, train_loader_q1, val_loader_q1,
#         epochs=CONFIG_Q1['epochs'], lr=CONFIG_Q1['lr'], weight_decay=CONFIG_Q1['weight_decay'],
#         save_dir=CONFIG_Q1['save_dir']
#     )
#     plot_history(history_alex)

print('AlexNet training cell ready (uncomment when dataset is set up).')

### 1-3-2. AlexNet Results and Evaluation (15 pts)
- Plot training/validation curves (loss + accuracy).
- Report test metrics: Accuracy, per-class Sensitivity, per-class Specificity.
- Show confusion matrix and discuss which classes are most/least correctly classified.

In [ ]:
# Q1: Evaluate AlexNet on test set
# ckpt_alex = torch.load(best_path_alex, map_location=device)
# model_alex.load_state_dict(ckpt_alex['model_state'])
# results_alex = evaluate_full(model_alex, test_loader_q1, class_names_q1)

print('AlexNet evaluation cell ready (uncomment when trained).')

### 1-4-1. Transfer Learning (VGG19) Implementation (20 pts)
- Use VGG19 pretrained on ImageNet; freeze early layers.
- Replace classifier head with new Dense layer for rice classes.
- Re-apply preprocessing with ImageNet mean/std for all splits.
- Train for 30 epochs (same as AlexNet for fair comparison).

In [ ]:
# Q1: Train VGG19 Transfer Learning
# CONFIG_Q1_TL = CONFIG_Q1.copy()
# CONFIG_Q1_TL['model'] = 'vgg19_tl'
# CONFIG_Q1_TL['freeze_backbone'] = True

# loaders_q1_tl = build_dataloaders(CONFIG_Q1_TL)  # Rebuilds with ImageNet norm
# if loaders_q1_tl:
#     train_loader_q1_tl, val_loader_q1_tl, test_loader_q1_tl, class_names_q1_tl = loaders_q1_tl
#     model_vgg = build_vgg19_tl(num_classes=len(class_names_q1_tl), freeze_backbone=True)
#     history_vgg, best_path_vgg = fit(
#         model_vgg, train_loader_q1_tl, val_loader_q1_tl,
#         epochs=CONFIG_Q1_TL['epochs'], lr=CONFIG_Q1_TL['lr'], weight_decay=CONFIG_Q1_TL['weight_decay'],
#         save_dir=CONFIG_Q1_TL['save_dir']
#     )
#     plot_history(history_vgg)

print('VGG19 TL training cell ready (uncomment when dataset is set up).')

### 1-4-2. Transfer Learning Results and Evaluation (20 pts)
- Report test metrics as above.
- Compare with AlexNet:
  - Data efficiency (accuracy gain with fewer epochs)?
  - How pretrained weights improved performance.
  - Architectural rationale: VGGNet + Inception/Global Pooling choices.
  - Why Global Pooling instead of FC layers?

#### Theoretical Analysis: Transfer Learning vs From-Scratch

**Data Efficiency**
- Transfer learning achieves higher accuracy with fewer epochs because pretrained weights on ImageNet already encode low-level features (edges, textures) and mid-level patterns (shapes, object parts) that generalize to new tasks.
- From-scratch training (AlexNet) must learn these representations from limited rice-disease data, requiring more epochs and larger datasets to converge.

**How Pretrained Weights Improve Performance**
1. **Feature reuse**: Early convolutional layers learn universal visual features (Gabor-like filters, edge detectors) that are task-agnostic. Freezing these layers prevents overfitting on small datasets.
2. **Faster convergence**: Only the final classifier layers adapt to rice-disease classes, reducing the effective parameter search space.
3. **Regularization effect**: Pretrained weights act as a strong prior, constraining the model to biologically plausible feature hierarchies.

**Architectural Rationale: VGG + Inception + Global Pooling**

*Why VGG19 as backbone?*
- VGG's uniform 3×3 convolutions with deep stacking (16-19 layers) build rich hierarchical features.
- Pretrained on ImageNet (1000 classes, 1.2M images), it captures diverse visual concepts transferable to agricultural imagery.

*Why add Inception-like blocks?*
$$
\text{Inception}(x) = \text{Concat}\Big[\text{Conv}_{1\times1}(x),\, \text{Conv}_{3\times3}(x),\, \text{Conv}_{5\times5}(x),\, \text{MaxPool}(x)\Big]
$$
- Multi-scale receptive fields capture both fine-grained leaf texture (small kernels) and global lesion patterns (large kernels) simultaneously.
- Reduces parameters via $1\times1$ bottlenecks before expensive $3\times3$ and $5\times5$ convolutions.

*Why Global Average Pooling (GAP) instead of Fully Connected layers?*
1. **Parameter reduction**: FC layers (e.g., $7\times7\times512 \to 4096$) introduce ~100M parameters, causing overfitting on small datasets. GAP averages each feature map to a scalar, adding zero parameters.
2. **Spatial invariance**: GAP enforces that class activation can occur anywhere in the feature map, improving robustness to object position/scale.
3. **Interpretation**: Each feature map directly corresponds to a class score, enabling class activation mapping (CAM) for visualization.

**Mathematical Formulation**
- GAP for feature map $f \in \mathbb{R}^{H \times W}$:
$$
\text{GAP}(f) = \frac{1}{HW}\sum_{i=1}^H\sum_{j=1}^W f_{ij}
$$
- Final classifier with $C$ classes and $K$ feature maps:
$$
y_c = \sum_{k=1}^K w_{c,k}\,\text{GAP}(f_k) + b_c,\quad \text{softmax}(\mathbf{y})
$$


In [ ]:
# Q1: Evaluate VGG19 TL on test set and compare
# ckpt_vgg = torch.load(best_path_vgg, map_location=device)
# model_vgg.load_state_dict(ckpt_vgg['model_state'])
# results_vgg = evaluate_full(model_vgg, test_loader_q1_tl, class_names_q1_tl)

# Comparison table (fill manually or programmatically):
# print('\nComparison AlexNet vs VGG19 TL:')
# print(f"AlexNet  - Test Acc: {results_alex['test_acc']:.4f}")
# print(f"VGG19 TL - Test Acc: {results_vgg['test_acc']:.4f}")

print('VGG19 TL evaluation cell ready (uncomment when trained).')

# Question 2 — Vehicle Classification in Road Scenes

> Paste the exact question prompt here if needed. Follow the rubric.

## 2-1. Dataset Preparation (10 pts)
- Prepare dataset and define classes (e.g., car, truck, bus, bike).
- Train/val/test split with clear distribution reporting.

## 2-2. Data Preprocessing (15 pts)
- Resize to 224×224 and normalize.
- Select augmentations suitable for road scenes; justify choices.

## 2-3. Implementation (20 pts)
- Implement baseline CNN or reuse `SimpleCNN` (or AlexNet) with tuned hyperparameters.
- Optionally add transfer learning (e.g., ResNet18/VGG19) for comparison.

## 2-4. Evaluation (30 pts)
- Plot curves and report metrics on the test set: Accuracy, per-class Recall/Sensitivity, per-class Specificity, confusion matrix.

## 2-5. Analysis and Optimization
### 2-5-1. Effect of Data Augmentation (10)
- Compare No-Aug vs Aug runs; summarize impact.

### 2-5-2. Role of Optimizer/Regularization (10)
- Compare optimizers (SGD vs AdamW) and/or weight decay/dropout settings.

### 2-5-3. Fine-tuning (10)
- Fine-tune deeper layers of the pretrained backbone; summarize gains/trade-offs.

#### Theoretical Answers: Q2 Analysis

The following subsections provide theoretical foundations for data augmentation, optimizer/regularization choices, and fine-tuning strategies in vehicle/land-use classification tasks.


## Q2 — Overview
- Goal: classify road-scene vehicles (e.g., car/truck/bus/bike).
- Baseline + optional transfer learning; follow Q1’s flow.
- Evaluate with accuracy, per-class sensitivity/specificity, and confusion matrix; analyze augmentations, optimizer/regularization, and fine-tuning.

### 2-1. Dataset Preparation (10 pts)
- Download UC-Merced Land Use dataset from Kaggle.
- Split 80/20 for train/test.
- Report class distributions and plot histogram.

In [ ]:
# Q2: Dataset Download (UC-Merced)
# Uncomment to download:
# import kagglehub
# path = kagglehub.dataset_download("abdulhasibuddin/uc-merced-land-use-dataset")
# print('Downloaded to:', path)

UC_MERCED_ROOT = Path('/path/to/uc-merced')  # UPDATE THIS
UC_MERCED_CLASSES = [  # 21 classes
    'agricultural', 'airplane', 'baseballdiamond', 'beach', 'buildings',
    'chaparral', 'denseresidential', 'forest', 'freeway', 'golfcourse',
    'harbor', 'intersection', 'mediumresidential', 'mobilehomepark', 'overpass',
    'parkinglot', 'river', 'runway', 'sparseresidential', 'storagetanks', 'tenniscourt'
]

if not UC_MERCED_ROOT.exists():
    print(f"UC-Merced not found at {UC_MERCED_ROOT}. Please download and update.")
else:
    print(f"UC-Merced root: {UC_MERCED_ROOT}")

In [ ]:
# Q2: Split UC-Merced 80/20
def split_ucmerced(src_root, dest_root, classes, train_r=0.8, seed=42):
    dest_root = Path(dest_root)
    for split in ['train', 'test']:
        for cls in classes:
            (dest_root / split / cls).mkdir(parents=True, exist_ok=True)
    
    counts = {'train': {}, 'test': {}}
    for cls in classes:
        cls_path = Path(src_root) / cls
        if not cls_path.exists():
            continue
        images = sorted(list(cls_path.glob('*.*')))
        train, test = train_test_split(images, train_size=train_r, random_state=seed)
        for img in train:
            shutil.copy(img, dest_root / 'train' / cls / img.name)
        for img in test:
            shutil.copy(img, dest_root / 'test' / cls / img.name)
        counts['train'][cls] = len(train)
        counts['test'][cls] = len(test)
    return counts

# UC_MERCED_SPLIT = Path('ucmerced_split')
# counts_ucm = split_ucmerced(UC_MERCED_ROOT, UC_MERCED_SPLIT, UC_MERCED_CLASSES)
# print('UC-Merced split counts:', counts_ucm)

print('UC-Merced split cell ready (uncomment when dataset is set up).')

### 2-2. Data Preprocessing (15 pts)
- Resize to 299×299 (for Xception).
- Normalize with ImageNet mean/std.
- Implement augmentations: rotation, zoom, horizontal/vertical flip.
- Justify choices for remote sensing imagery.
- Show augmented examples.

In [ ]:
# Q2: Xception preprocessing (299x299, ImageNet norm, augmentations)
IMAGENET_MEAN_Q2 = [0.485, 0.456, 0.406]
IMAGENET_STD_Q2 = [0.229, 0.224, 0.225]

def get_transforms_q2(img_size=299):
    train_tfms = transforms.Compose([
        transforms.Resize(int(img_size * 1.15)),
        transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
        transforms.RandomRotation(45),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN_Q2, IMAGENET_STD_Q2)
    ])
    test_tfms = transforms.Compose([
        transforms.Resize(int(img_size * 1.15)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN_Q2, IMAGENET_STD_Q2)
    ])
    return train_tfms, test_tfms

# Show augmented samples (similar to Q1)
# train_tfms_q2, _ = get_transforms_q2()
# show_augmented_samples(UC_MERCED_SPLIT / 'train', UC_MERCED_CLASSES[0], train_tfms_q2)

print('Q2 transforms ready.')

### 2-3. Implementation (20 pts)
- Feature extractor: Xception pretrained on ImageNet (frozen).
- Classifier: Enhanced MLP (Normalization → Sigmoid → Dropout(0.4) → Softmax).
- Optimizer: Adagrad for MLP.
- Train for 50 epochs; report architecture and hyperparameters.

In [ ]:
# Q2: Xception + Enhanced MLP (CNN-MLP from paper)
# Note: PyTorch doesn't have Xception in torchvision; use timm or define manually
# For demo, we use a placeholder approach with a pretrained model

try:
    import timm
    HAS_TIMM = True
except:
    HAS_TIMM = False
    print('timm not installed. Install via: pip install timm')

class XceptionMLP(nn.Module):
    def __init__(self, num_classes=21, dropout=0.4):
        super().__init__()
        if HAS_TIMM:
            self.feature_extractor = timm.create_model('xception', pretrained=True, num_classes=0)
            for p in self.feature_extractor.parameters():
                p.requires_grad = False
            feat_dim = self.feature_extractor.num_features
        else:
            # Fallback: use ResNet50 as substitute
            base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
            self.feature_extractor = nn.Sequential(*list(base.children())[:-1])
            for p in self.feature_extractor.parameters():
                p.requires_grad = False
            feat_dim = 2048
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(feat_dim),
            nn.Sigmoid(),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, num_classes)
        )
    
    def forward(self, x):
        feats = self.feature_extractor(x)
        if feats.dim() == 4:
            feats = self.global_avg_pool(feats)
        feats = feats.view(feats.size(0), -1)
        out = self.mlp(feats)
        return out

# model_q2 = XceptionMLP(num_classes=len(UC_MERCED_CLASSES)).to(device)
# print(model_q2)

print('Xception-MLP model class ready.')

In [ ]:
# Q2: Train Xception-MLP with Adagrad
# def fit_adagrad(model, train_loader, val_loader, epochs, lr, save_dir):
#     criterion = nn.CrossEntropyLoss()
#     # Adagrad optimizer for MLP only
#     mlp_params = [p for n, p in model.named_parameters() if 'mlp' in n and p.requires_grad]
#     optimizer = optim.Adagrad(mlp_params, lr=lr)
#     
#     history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
#     best_acc, best_path = -1.0, Path(save_dir) / 'best_xception_mlp.pth'
#     
#     for e in range(1, epochs + 1):
#         t0 = time.time()
#         tr_l, tr_a = train_one_epoch(model, train_loader, criterion, optimizer)
#         va_l, va_a, _, _ = evaluate(model, val_loader, criterion)
#         history['train_loss'].append(tr_l)
#         history['train_acc'].append(tr_a)
#         history['val_loss'].append(va_l)
#         history['val_acc'].append(va_a)
#         if va_a > best_acc:
#             best_acc = va_a
#             torch.save({'model_state': model.state_dict()}, str(best_path))
#         print(f"Epoch {e:03d}/{epochs} | train {tr_l:.4f}/{tr_a:.4f} | val {va_l:.4f}/{va_a:.4f} | {time.time()-t0:.1f}s")
#     print(f"Best val acc: {best_acc:.4f}")
#     return history, best_path

# train_tfms_q2, test_tfms_q2 = get_transforms_q2()
# train_ds_q2 = datasets.ImageFolder(str(UC_MERCED_SPLIT / 'train'), transform=train_tfms_q2)
# test_ds_q2 = datasets.ImageFolder(str(UC_MERCED_SPLIT / 'test'), transform=test_tfms_q2)
# train_loader_q2 = DataLoader(train_ds_q2, batch_size=32, shuffle=True, num_workers=2)
# test_loader_q2 = DataLoader(test_ds_q2, batch_size=32, shuffle=False, num_workers=2)

# model_q2 = XceptionMLP(num_classes=len(UC_MERCED_CLASSES)).to(device)
# history_q2, best_path_q2 = fit_adagrad(model_q2, train_loader_q2, test_loader_q2, epochs=50, lr=0.01, save_dir='outputs')
# plot_history(history_q2)

print('Q2 training cell ready (uncomment when dataset is set up).')

### 2-4. Evaluation (30 pts)
- Plot train/val curves.
- Report confusion matrix on test set.
- Discuss most/least correctly classified classes.
- Compare with paper's CNN-MLP: data efficiency?
- Why use separate MLP instead of FC layers at CNN end?
- Role of Global Average Pooling before MLP?
- Why Adagrad for MLP?

In [ ]:
# Q2: Evaluate Xception-MLP
# ckpt_q2 = torch.load(best_path_q2, map_location=device)
# model_q2.load_state_dict(ckpt_q2['model_state'])
# results_q2 = evaluate_full(model_q2, test_loader_q2, UC_MERCED_CLASSES)

print('Q2 evaluation cell ready (uncomment when trained).')

### 2-5-1. Effect of Data Augmentation (10 pts)
- Retrain CNN-MLP without augmentations (only resize + normalize).
- Compare final test accuracy with augmented version.
- Report in table and discuss impact.

In [ ]:
# Q2: Ablation — No Augmentation
# def get_transforms_q2_noaug(img_size=299):
#     tfms = transforms.Compose([
#         transforms.Resize(int(img_size * 1.15)),
#         transforms.CenterCrop(img_size),
#         transforms.ToTensor(),
#         transforms.Normalize(IMAGENET_MEAN_Q2, IMAGENET_STD_Q2)
#     ])
#     return tfms, tfms

# train_tfms_noaug, test_tfms_noaug = get_transforms_q2_noaug()
# train_ds_noaug = datasets.ImageFolder(str(UC_MERCED_SPLIT / 'train'), transform=train_tfms_noaug)
# train_loader_noaug = DataLoader(train_ds_noaug, batch_size=32, shuffle=True, num_workers=2)

# model_q2_noaug = XceptionMLP(num_classes=len(UC_MERCED_CLASSES)).to(device)
# history_noaug, best_path_noaug = fit_adagrad(model_q2_noaug, train_loader_noaug, test_loader_q2, epochs=50, lr=0.01, save_dir='outputs')
# plot_history(history_noaug)

# ckpt_noaug = torch.load(best_path_noaug, map_location=device)
# model_q2_noaug.load_state_dict(ckpt_noaug['model_state'])
# results_noaug = evaluate_full(model_q2_noaug, test_loader_q2, UC_MERCED_CLASSES)

# print(f"With Aug: {results_q2['test_acc']:.4f} | Without Aug: {results_noaug['test_acc']:.4f}")

print('No-aug ablation cell ready (uncomment when baseline trained).')

### 2-5-2. Role of Optimizer (10 pts)
- Replace Adagrad with Adam; retrain with same settings (including augmentation).
- Compare final test accuracy.
- Discuss: Is Adagrad critical for performance or is Adam comparable/better?

In [ ]:
# Q2: Ablation — Adam Optimizer
# def fit_adam(model, train_loader, val_loader, epochs, lr, save_dir):
#     criterion = nn.CrossEntropyLoss()
#     mlp_params = [p for n, p in model.named_parameters() if 'mlp' in n and p.requires_grad]
#     optimizer = optim.Adam(mlp_params, lr=lr)
#     history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
#     best_acc, best_path = -1.0, Path(save_dir) / 'best_xception_mlp_adam.pth'
#     for e in range(1, epochs + 1):
#         t0 = time.time()
#         tr_l, tr_a = train_one_epoch(model, train_loader, criterion, optimizer)
#         va_l, va_a, _, _ = evaluate(model, val_loader, criterion)
#         history['train_loss'].append(tr_l)
#         history['train_acc'].append(tr_a)
#         history['val_loss'].append(va_l)
#         history['val_acc'].append(va_a)
#         if va_a > best_acc:
#             best_acc = va_a
#             torch.save({'model_state': model.state_dict()}, str(best_path))
#         print(f"Epoch {e:03d}/{epochs} | train {tr_l:.4f}/{tr_a:.4f} | val {va_l:.4f}/{va_a:.4f} | {time.time()-t0:.1f}s")
#     print(f"Best val acc: {best_acc:.4f}")
#     return history, best_path

# model_q2_adam = XceptionMLP(num_classes=len(UC_MERCED_CLASSES)).to(device)
# history_adam, best_path_adam = fit_adam(model_q2_adam, train_loader_q2, test_loader_q2, epochs=50, lr=0.001, save_dir='outputs')
# plot_history(history_adam)

# ckpt_adam = torch.load(best_path_adam, map_location=device)
# model_q2_adam.load_state_dict(ckpt_adam['model_state'])
# results_adam = evaluate_full(model_q2_adam, test_loader_q2, UC_MERCED_CLASSES)

# print(f"Adagrad: {results_q2['test_acc']:.4f} | Adam: {results_adam['test_acc']:.4f}")

print('Adam optimizer ablation cell ready (uncomment when baseline trained).')

### 2-5-3. Fine-tuning (10 pts)

Unfreeze last 10 layers of Xception backbone and fine-tune with very low learning rate.

In [ ]:
# Q2: Fine-tuning with unfrozen layers
# def unfreeze_last_n_layers(model, n=10):
#     """Unfreeze last n layers of Xception backbone."""
#     layers = list(model.backbone.children())
#     for layer in layers[:-n]:
#         for param in layer.parameters():
#             param.requires_grad = False
#     for layer in layers[-n:]:
#         for param in layer.parameters():
#             param.requires_grad = True

# def fit_finetuned(model, train_loader, val_loader, epochs, lr, save_dir):
#     criterion = nn.CrossEntropyLoss()
#     params_to_optimize = [p for p in model.parameters() if p.requires_grad]
#     optimizer = optim.Adam(params_to_optimize, lr=lr, weight_decay=1e-4)
#     scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
#     history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
#     best_acc, best_path = -1.0, Path(save_dir) / 'best_xception_mlp_finetuned.pth'
#     for e in range(1, epochs + 1):
#         t0 = time.time()
#         tr_l, tr_a = train_one_epoch(model, train_loader, criterion, optimizer)
#         scheduler.step()
#         va_l, va_a, _, _ = evaluate(model, val_loader, criterion)
#         history['train_loss'].append(tr_l)
#         history['train_acc'].append(tr_a)
#         history['val_loss'].append(va_l)
#         history['val_acc'].append(va_a)
#         if va_a > best_acc:
#             best_acc = va_a
#             torch.save({'model_state': model.state_dict()}, str(best_path))
#         print(f"Epoch {e:03d}/{epochs} | train {tr_l:.4f}/{tr_a:.4f} | val {va_l:.4f}/{va_a:.4f} | {time.time()-t0:.1f}s")
#     print(f"Best val acc: {best_acc:.4f}")
#     return history, best_path

# model_q2_ft = XceptionMLP(num_classes=len(UC_MERCED_CLASSES)).to(device)
# unfreeze_last_n_layers(model_q2_ft, n=10)
# history_ft, best_path_ft = fit_finetuned(model_q2_ft, train_loader_q2, test_loader_q2, epochs=30, lr=1e-5, save_dir='outputs')
# plot_history(history_ft)

# ckpt_ft = torch.load(best_path_ft, map_location=device)
# model_q2_ft.load_state_dict(ckpt_ft['model_state'])
# results_ft = evaluate_full(model_q2_ft, test_loader_q2, UC_MERCED_CLASSES)

# print(f"Frozen: {results_q2['test_acc']:.4f} | Fine-tuned: {results_ft['test_acc']:.4f}")

print('Fine-tuning ablation cell ready (uncomment when baseline trained).')

---

## 3. Results Summary

Below are the final results for both questions.

### 3-1. Q1 Results: AlexNet vs VGG19 Transfer Learning

| Model | Test Accuracy | Per-Class Sensitivity | Per-Class Specificity |
|-------|---------------|----------------------|----------------------|
| AlexNet (from scratch) | - | - | - |
| VGG19 Transfer Learning | - | - | - |

**Note:** Fill in results after training both models.

### 3-2. Q2 Results: Ablation Studies

| Experiment | Test Accuracy | Notes |
|-----------|---------------|-------|
| Baseline (Xception+MLP, Adagrad) | - | With augmentation |
| No Augmentation | - | Compare augmentation impact |
| Adam Optimizer | - | Compare optimizer |
| Fine-tuned (10 layers) | - | Unfreeze last 10 layers |

**Note:** Fill in results after completing all ablation studies.

---

## 4. Submission Checklist

- [ ] Q1: Rice Leaf Disease dataset prepared (70/15/15 split)
- [ ] Q1: AlexNet trained from scratch and evaluated
- [ ] Q1: VGG19 transfer learning trained and evaluated
- [ ] Q1: Comparison table and analysis completed
- [ ] Q2: UC-Merced dataset prepared (80/20 split)
- [ ] Q2: Xception+MLP baseline trained with Adagrad
- [ ] Q2: Ablation 1 (no augmentation) completed
- [ ] Q2: Ablation 2 (Adam optimizer) completed
- [ ] Q2: Ablation 3 (fine-tuning) completed
- [ ] Q2: Results table and analysis completed
- [ ] All confusion matrices and visualizations generated
- [ ] Final results summary filled in

# Submission Checklist
- [ ] All prompts pasted and answered under each section.
- [ ] Dataset paths configured and splits verified.
- [ ] Model choices justified (baseline vs transfer learning).
- [ ] Hyperparameters listed (epochs, LR, batch size, weight decay).
- [ ] Training curves included.
- [ ] Final test metrics reported (Accuracy, Sensitivity, Specificity).
- [ ] Confusion matrices included and discussed.
- [ ] Error analysis and discussion added.
- [ ] References cited.